# IndexTTS-DubFlow（全程可见日志版）

基于 IndexTTS 2.5 的原声视频翻译与智能配音工作流。

这个 Notebook 自带完整项目源码，可在 Colab 中启动 Gradio Web 面板。支持上传、路径、Google Drive、对白/背景分离、歌曲人声保护、多 ASR、热词库、字幕导入、人工译文锁定、智能断句、Index-Translate / Homura、HY-MT2-7B 上下文翻译、原音色/情感配音、自然语速优先时间规划、时间轴导入导出和视频输出。

**Web 构建：2026.10.05-r8-index**。如果打开面板后没有看到 Qwen3/FunASR/FireRedASR2、Index 翻译选项，请确认运行的是本 Notebook，而不是旧标签页。

> 安装与下载每 10 秒至少输出一次状态；Web 启动每 5 秒输出一次状态。任何失败都会显示日志尾部。

> 推荐 L4/A100/A10。T4 不支持 BF16，程序会自动切到 FP32，速度更慢且可能显存不足。

> 已跟进 IndexTTS 2.5 正式合并至官方 `main` 的提交 `ccd81054de9859faeb19b773fff0e2e1ae9e959e`。安装器会验证模型目录中的 2.5 专属结构，并修复官方 Hub `config.yaml` 当前错误的 2.0 标记与 `/cubefs` 内部路径；真实的 2.0 配置仍会被拒绝。

R8 新增 Index-Translate 官方公网 API 和自建服务、Homura 音节控制。启动后在高级设置选择；公网模式无需翻译 GPU，但会发送原文、上下文和术语。自建服务需自行部署。详见 `docs/INDEX_TRANSLATE.md`。

## 1. 运行选项

In [ ]:
import os, secrets
MOUNT_DRIVE = True              # 是否挂载 Google Drive
MODEL_SOURCE = 'modelscope'     # 'modelscope' 或 'huggingface'
INSTALL_DEMUCS = True           # 推荐：分离原对白与背景
INSTALL_SINGING_DETECTOR = True # 保留歌曲人声，只从背景移除对白
INSTALL_DIARIZATION = False     # 多人物自动分离；需要 HF token，安装更慢
PERSIST_MODEL_CACHE = False     # True 会把模型缓存在云盘（约需 10GB+，以后重开免下载）
HYMT_DEFAULT_MODEL = 'tencent/Hy-MT2-7B' # 7B 质量更高；1.8B 约省 12GB 磁盘
HYMT_CACHE_ON_DRIVE = False     # True 仅把 HY-MT2 权重放到 Drive，保留本地 Qwen ASR
CLEAN_INSTALLER_CACHE = True    # 安装成功后回收 uv/pip 下载缓存，不影响已安装环境
DISABLE_HF_XET = True           # 避免大模型下载产生额外 Xet 分块/重建峰值
CREATE_PUBLIC_LINK = False      # 默认使用更快、更稳的 Colab 私有代理；True 再建 gradio.live 链接
LOG_HEARTBEAT_SECONDS = 10      # 安装静默时每多少秒输出一次心跳（5～60）
WEB_START_TIMEOUT = 180         # 本地 Web 服务启动超时（秒）
GRADIO_AUTH = globals().get('GRADIO_AUTH') or ('dubber:' + secrets.token_urlsafe(12))
if PERSIST_MODEL_CACHE and not MOUNT_DRIVE:
    raise ValueError('PERSIST_MODEL_CACHE=True 时必须同时开启 MOUNT_DRIVE')
if HYMT_CACHE_ON_DRIVE and not MOUNT_DRIVE:
    raise ValueError('HYMT_CACHE_ON_DRIVE=True 时必须同时开启 MOUNT_DRIVE')
MODEL_DIR = (
    '/content/drive/MyDrive/IndexTTS25_Cache/checkpoints'
    if PERSIST_MODEL_CACHE else '/content/index-tts/checkpoints'
)
RUNTIME_CACHE_DIR = (
    '/content/drive/MyDrive/IndexTTS25_Cache/runtime'
    if PERSIST_MODEL_CACHE else '/content/IndexTTS25_Runtime_Cache'
)
ASR_RUNTIME_DIR = (
    '/content/drive/MyDrive/IndexTTS25_Cache/asr_runtimes'
    if PERSIST_MODEL_CACHE else '/content/IndexTTS25_ASR_Runtimes'
)
TRANSLATION_RUNTIME_DIR = (
    '/content/drive/MyDrive/IndexTTS25_Cache/translation_runtimes'
    if PERSIST_MODEL_CACHE else '/content/IndexTTS25_Translation_Runtimes'
)
HYMT_HF_HOME = (
    '/content/drive/MyDrive/IndexTTS25_Cache/hymt_huggingface'
    if HYMT_CACHE_ON_DRIVE else os.path.join(RUNTIME_CACHE_DIR, 'huggingface')
)
print('Web 面板登录凭据：', GRADIO_AUTH)
print('模型目录：', MODEL_DIR)
print('可选 ASR 缓存：', ASR_RUNTIME_DIR)
print('HY-MT2 运行时缓存：', TRANSLATION_RUNTIME_DIR)
print('HY-MT2 权重缓存：', HYMT_HF_HOME)
print('HY-MT2 默认模型：', HYMT_DEFAULT_MODEL)
print('安装日志心跳：每', LOG_HEARTBEAT_SECONDS, '秒')
print('公网链接：', '启用' if CREATE_PUBLIC_LINK else '关闭（使用 Colab 私有链接）')


## 2. 解包内嵌项目源码

In [ ]:
import base64, io, shutil, zipfile
from pathlib import Path

BUNDLE_B64 = '''...'''
workspace = Path('/content')
project = workspace / 'IndexTTS-DubFlow'
if project.is_dir():
    shutil.rmtree(project)
    print('已移除旧项目源码，防止旧 Web/UI 缓存混入。')
with zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE_B64))) as archive:
    archive.extractall(workspace)
print('项目已解包：', project)


## 3. 挂载 Google Drive（可选）

In [ ]:
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('跳过 Google Drive')


## 4. 检查 GPU

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=False)


## 5. 安装环境并下载模型

首次运行会创建独立 Python 3.11/uv 环境并下载数 GB 权重。每个阶段都会显示耗时，静默命令每 10 秒也会显示一次心跳和目录大小；全部输出同时保存到 `/content/indextts25_install.log`，中途失败可直接重跑续传。

In [ ]:
import os, queue, subprocess, threading, time
from pathlib import Path
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['INDEXTTS_DIR'] = '/content/index-tts'
env['INDEXTTS_MODEL_DIR'] = MODEL_DIR
env['MODEL_SOURCE'] = MODEL_SOURCE
env['INSTALL_DEMUCS'] = '1' if INSTALL_DEMUCS else '0'
env['INSTALL_SINGING_DETECTOR'] = '1' if INSTALL_SINGING_DETECTOR else '0'
env['INSTALL_DIARIZATION'] = '1' if INSTALL_DIARIZATION else '0'
env['CLEAN_INSTALLER_CACHE'] = '1' if CLEAN_INSTALLER_CACHE else '0'
env['LOG_HEARTBEAT_SECONDS'] = str(LOG_HEARTBEAT_SECONDS)
env['HF_HOME'] = os.path.join(RUNTIME_CACHE_DIR, 'huggingface')
env['HF_HUB_DISABLE_XET'] = '1' if DISABLE_HF_XET else '0'
env['TORCH_HOME'] = os.path.join(RUNTIME_CACHE_DIR, 'torch')
env['UV_CACHE_DIR'] = '/content/IndexTTS25_Installer_Cache/uv'
env['PIP_CACHE_DIR'] = '/content/IndexTTS25_Installer_Cache/pip'
install_log = Path('/content/indextts25_install.log')
started = time.monotonic()
print('=' * 72, flush=True)
print('开始安装：下面会持续输出阶段、耗时、目录大小和下载进度。', flush=True)
print('完整日志：', install_log, flush=True)
print('=' * 72, flush=True)
process = subprocess.Popen(
    ['bash', '/content/IndexTTS-DubFlow/scripts/colab_setup.sh'],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, encoding='utf-8', errors='replace', bufsize=1,
)
lines = queue.Queue()
def _read_install_output():
    assert process.stdout is not None
    for line in iter(process.stdout.readline, ''):
        lines.put(line)
    process.stdout.close()
reader = threading.Thread(target=_read_install_output, daemon=True)
reader.start()
last_output = time.monotonic()
with install_log.open('w', encoding='utf-8', buffering=1) as log_handle:
    try:
        while process.poll() is None or reader.is_alive() or not lines.empty():
            try:
                line = lines.get(timeout=1)
            except queue.Empty:
                line = ''
            if line:
                print(line, end='', flush=True)
                log_handle.write(line)
                last_output = time.monotonic()
            elif process.poll() is None and time.monotonic() - last_output >= LOG_HEARTBEAT_SECONDS:
                elapsed = (time.monotonic() - started) / 60
                heartbeat = f'[Notebook] ⏳ 安装进程仍在运行，总耗时 {elapsed:.1f} 分钟\n'
                print(heartbeat, end='', flush=True)
                log_handle.write(heartbeat)
                last_output = time.monotonic()
    except KeyboardInterrupt:
        process.terminate()
        print('已停止本次安装；重新运行此单元会继续使用已下载内容。', flush=True)
        raise
return_code = process.wait()
if return_code != 0:
    tail = install_log.read_text(encoding='utf-8', errors='replace')[-12000:]
    raise RuntimeError(f'安装失败（退出码 {return_code}）。最后日志：\n{tail}')
print(f'安装单元完成，总耗时 {(time.monotonic() - started) / 60:.1f} 分钟。', flush=True)


## 5.1 磁盘诊断与安全清理（下载失败后运行）

只回收可重建的 Xet、uv、pip 临时缓存；不会删除 IndexTTS、Qwen ASR、ForcedAligner、HY-MT2 已完成的 Hub 分片、视频或输出。出现 `No space left on device` 时运行一次，再重启 Web 重试。

In [ ]:
import os, shutil, subprocess
from pathlib import Path

AUTO_CLEAN_TEMP_CACHE = True
GIB = 1024 ** 3
def _allocated(path):
    if not path.exists():
        return 0
    total = 0
    seen = set()
    for item in path.rglob('*'):
        try:
            stat = item.stat()
        except OSError:
            continue
        if item.is_file() and (stat.st_dev, stat.st_ino) not in seen:
            seen.add((stat.st_dev, stat.st_ino))
            total += stat.st_blocks * 512 if stat.st_blocks else stat.st_size
    return total

targets = [
    Path(HYMT_HF_HOME) / 'xet',
    Path(RUNTIME_CACHE_DIR) / 'huggingface' / 'xet',
    Path('/root/.cache/huggingface/xet'),
    Path('/root/.cache/uv'),
    Path('/root/.cache/pip'),
    Path('/content/IndexTTS25_Installer_Cache'),
]
unique_targets = list(dict.fromkeys(path.resolve() for path in targets))
reclaimed = 0
for target in unique_targets:
    size = _allocated(target)
    if size:
        print(f'可回收：{target} = {size/GIB:.2f} GiB')
        if AUTO_CLEAN_TEMP_CACHE:
            shutil.rmtree(target)
            reclaimed += size
os.environ['HF_HUB_DISABLE_XET'] = '1' if DISABLE_HF_XET else '0'
print(f'✅ 已回收 {reclaimed/GIB:.2f} GiB 临时缓存。模型和已完成分片均保留。')
subprocess.run(['df', '-h', '/content'], check=False)
for path in [
    '/content/index-tts', ASR_RUNTIME_DIR, TRANSLATION_RUNTIME_DIR,
    str(Path(RUNTIME_CACHE_DIR) / 'huggingface'), HYMT_HF_HOME,
]:
    if Path(path).exists():
        subprocess.run(['du', '-sh', path], check=False)


## 6. 启动 Web 面板

该单元会把 Web 服务放到后台，探测成功后立即显示 **Colab 私有链接**，不再无限等待 Gradio 公网隧道。单元运行结束不代表面板关闭；可用下一单元查看日志或停止服务。

In [ ]:
import html, json, os, re, socket, subprocess, time, urllib.error, urllib.request
from pathlib import Path
from IPython.display import HTML, display

def _free_port(start=7860, stop=7870):
    for candidate in range(start, stop + 1):
        with socket.socket() as sock:
            try:
                sock.bind(('127.0.0.1', candidate))
            except OSError:
                continue
            return candidate
    raise RuntimeError(f'{start}-{stop} 端口都被占用，请先运行下面的状态/停止单元。')

def _http_ready(port):
    try:
        with urllib.request.urlopen(f'http://127.0.0.1:{port}/', timeout=2) as response:
            return 200 <= response.status < 500
    except urllib.error.HTTPError as exc:
        return exc.code in (401, 403)
    except Exception:
        return False

web_port = _free_port()
web_log = Path('/content/indextts25_web.log')
startup_info = Path('/content/indextts25_web_status.json')
web_log.write_text('', encoding='utf-8')
startup_info.unlink(missing_ok=True)
env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['PYTHONIOENCODING'] = 'utf-8'
env['GRADIO_ANALYTICS_ENABLED'] = 'False'
env['INDEXTTS_DIR'] = '/content/index-tts'
env['INDEXTTS_MODEL_DIR'] = MODEL_DIR
env['INDEXTTS_CONFIG'] = os.path.join(MODEL_DIR, 'config.yaml')
env['MODEL_SOURCE'] = MODEL_SOURCE
env['ASR_RUNTIME_ROOT'] = ASR_RUNTIME_DIR
env['TRANSLATION_RUNTIME_ROOT'] = TRANSLATION_RUNTIME_DIR
env['HF_HOME'] = os.path.join(RUNTIME_CACHE_DIR, 'huggingface')
env['HYMT_HF_HOME'] = HYMT_HF_HOME
env['HYMT_DEFAULT_MODEL'] = HYMT_DEFAULT_MODEL
env['HF_HUB_DISABLE_XET'] = '1' if DISABLE_HF_XET else '0'
env['PIP_NO_CACHE_DIR'] = '1'
env['TORCH_HOME'] = os.path.join(RUNTIME_CACHE_DIR, 'torch')
env['DUBBER_OUTPUT_ROOT'] = (
    '/content/drive/MyDrive/IndexTTS_Dubber_Outputs'
    if MOUNT_DRIVE else '/content/IndexTTS_Dubber_Outputs'
)
env['SEPARATE_BACKGROUND_DEFAULT'] = '1' if INSTALL_DEMUCS else '0'
env['PROTECT_SINGING_DEFAULT'] = '1' if INSTALL_SINGING_DETECTOR else '0'
if GRADIO_AUTH:
    env['GRADIO_AUTH'] = GRADIO_AUTH
command = [
    '/content/index-tts/.venv/bin/python', '-u',
    '/content/IndexTTS-DubFlow/app.py',
    '--server-name', '0.0.0.0', '--server-port', str(web_port),
    '--startup-info', str(startup_info),
]
if CREATE_PUBLIC_LINK:
    command.append('--share')
_DUBBER_LOG_HANDLE = web_log.open('w', encoding='utf-8', buffering=1)
_DUBBER_WEB_PROCESS = subprocess.Popen(
    command, env=env, stdout=_DUBBER_LOG_HANDLE, stderr=subprocess.STDOUT,
    start_new_session=True,
)
_DUBBER_WEB_PORT = web_port
Path('/content/indextts25_web.pid').write_text(str(_DUBBER_WEB_PROCESS.pid), encoding='ascii')
print(f'Web 子进程 PID={_DUBBER_WEB_PROCESS.pid}，端口={web_port}', flush=True)
deadline = time.monotonic() + WEB_START_TIMEOUT
next_heartbeat = 0.0
last_log_position = 0
while time.monotonic() < deadline:
    if web_log.exists():
        with web_log.open('r', encoding='utf-8', errors='replace') as handle:
            handle.seek(last_log_position)
            fresh = handle.read()
            last_log_position = handle.tell()
        if fresh:
            print(fresh, end='', flush=True)
    return_code = _DUBBER_WEB_PROCESS.poll()
    if return_code is not None:
        tail = web_log.read_text(encoding='utf-8', errors='replace')[-8000:]
        raise RuntimeError(f'Web 进程提前退出（{return_code}）。最后日志：\n{tail}')
    if _http_ready(web_port):
        break
    now = time.monotonic()
    if now >= next_heartbeat:
        stage = '等待启动信息'
        if startup_info.exists():
            try:
                stage = json.loads(startup_info.read_text(encoding='utf-8')).get('message', stage)
            except Exception:
                pass
        elapsed = WEB_START_TIMEOUT - max(0, int(deadline - now))
        print(f'⏳ Web 仍在启动：{stage}（已等待 {elapsed} 秒）', flush=True)
        next_heartbeat = now + 5
    time.sleep(0.5)
else:
    tail = web_log.read_text(encoding='utf-8', errors='replace')[-8000:]
    _DUBBER_WEB_PROCESS.terminate()
    raise TimeoutError(f'{WEB_START_TIMEOUT} 秒内 Web 端口未就绪。最后日志：\n{tail}')
print('✅ 本地 Web 服务已通过 HTTP 探活；此时尚未加载 IndexTTS 权重。', flush=True)
try:
    from google.colab.output import eval_js
    proxy_url = eval_js(f'google.colab.kernel.proxyPort({web_port})')
    display(HTML(
        '<p><a style="font-size:20px;font-weight:700" target="_blank" href="'
        + html.escape(proxy_url, quote=True) + '">打开 IndexTTS-DubFlow 面板</a></p>'
    ))
    print('Colab 私有链接：', proxy_url, flush=True)
except Exception as exc:
    proxy_url = ''
    print('无法生成 Colab 私有代理链接：', exc, flush=True)
print('登录凭据：', GRADIO_AUTH, flush=True)
if CREATE_PUBLIC_LINK:
    print('本地面板已经可用，继续等待可选的 gradio.live 公网链接（最多 90 秒）…', flush=True)
    public_deadline = time.monotonic() + 90
    public_url = ''
    while time.monotonic() < public_deadline and _DUBBER_WEB_PROCESS.poll() is None:
        content = web_log.read_text(encoding='utf-8', errors='replace')
        match = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', content)
        if match:
            public_url = match.group(0)
            break
        time.sleep(1)
    if public_url:
        display(HTML('<p><a target="_blank" href="' + html.escape(public_url, quote=True) + '">打开公网 Gradio 链接</a></p>'))
        print('公网链接：', public_url, flush=True)
    else:
        print('⚠️ 公网隧道未在 90 秒内返回；上面的 Colab 私有链接仍可正常使用。', flush=True)


## 7. 查看日志或停止 Web（可选）

默认只查看状态。要关闭后台面板，把 `STOP_WEB` 改成 `True` 后运行。

In [ ]:
from pathlib import Path
import json
STOP_WEB = False
process = globals().get('_DUBBER_WEB_PROCESS')
if STOP_WEB:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except Exception:
            process.kill()
        print('Web 面板已停止。')
    else:
        print('当前没有由本 Notebook 启动的 Web 进程。')
else:
    status_path = Path('/content/indextts25_web_status.json')
    log_path = Path('/content/indextts25_web.log')
    if status_path.exists():
        print(json.dumps(json.loads(status_path.read_text(encoding='utf-8')), ensure_ascii=False, indent=2))
    print('\n最近日志：')
    print(log_path.read_text(encoding='utf-8', errors='replace')[-8000:] if log_path.exists() else '暂无日志')


## 8. 实时跟随 Web 后台日志（可选）

在面板中开始分析或生成后，可运行这个单元观察后台日志。默认跟随 10 分钟；按 Colab 的停止按钮只会停止日志跟随，不会停止 Web 面板或正在执行的任务。

In [ ]:
from pathlib import Path
import time
FOLLOW_SECONDS = 600
log_path = Path('/content/indextts25_web.log')
position = 0
started = time.monotonic()
next_heartbeat = 0.0
print(f'开始跟随 {log_path}，最长 {FOLLOW_SECONDS} 秒。', flush=True)
try:
    while time.monotonic() - started < FOLLOW_SECONDS:
        if log_path.exists():
            with log_path.open('r', encoding='utf-8', errors='replace') as handle:
                handle.seek(position)
                fresh = handle.read()
                position = handle.tell()
            if fresh:
                print(fresh, end='', flush=True)
        now = time.monotonic()
        if now >= next_heartbeat:
            process = globals().get('_DUBBER_WEB_PROCESS')
            state = '运行中' if process is not None and process.poll() is None else '进程状态未知或已停止'
            print(f'[Notebook] ⏳ Web {state}，日志跟随已运行 {int(now-started)} 秒', flush=True)
            next_heartbeat = now + 10
        time.sleep(1)
except KeyboardInterrupt:
    print('已停止日志跟随；Web 面板不受影响。', flush=True)


## 使用提醒

- 必须拥有视频、人物声音和翻译/输出用途的合法授权。
- 多人物视频可安装 Pyannote，或在时间轴中手工修改说话人标签。
- 默认检测并保留歌曲人声；检测到唱歌的片段会在时间轴中禁用翻译。对白与歌声完全重叠时仍需人工复核。
- 可上传 SRT/VTT/ASS/SSA 作为原文或最终译文；最终译文会直接锁定，不会重复翻译。
- 分析后可导出 JSON + 原文/译文 SRT；人工校对后可把 JSON/SRT/VTT/ASS/SSA 重新上传并锁定。
- FireRed、Qwen3-ASR、Fun-ASR-Nano 首次被选择时才建立隔离环境并下载模型，进度写入 Web 日志。
- HY-MT2-7B 原始权重为 16.1 GB；T4 的 4-bit 只省显存。R7 默认禁用 Xet 临时重建，下载前会做空间预检。
- 磁盘仍紧张时可把 HYMT_CACHE_ON_DRIVE=True，或在 Web 选择官方 Hy-MT2-1.8B（约 4 GB）；Qwen ASR/Aligner 无需删除。
- ASR 会尝试合并被截断的半句，并优先按句末标点、分句标点和静音切分；上传字幕不自动改动时间轴。
- TTS 先保留自然语速并借用下一句前的真实空白；放不下时才压缩，警告只提示明显压缩的句子。
- 逐句总时长严格一致不等于逐音素唇形同步。
- 详细架构、当前官方分支问题和许可证见项目 `README.md`。